SoundWall converter. This python script will grab data from the spreadsheets datavv.xlsx, datacc.xlsx, datavv2.xlsx and datacc2.xlsx and produce a json file that will construct the soundwall interactive.

In [1]:
import pandas as pd

def excel_to_json_string(input_file, output_file):
    # Read the Excel file
    excel_file = pd.ExcelFile(input_file)
    all_data = []

    # Iterate through each sheet in the Excel file
    for sheet_index, sheet_name in enumerate(excel_file.sheet_names):
        df = pd.read_excel(excel_file, sheet_name=sheet_name)
        sheet_suffix = f"{sheet_index + 1:02d}"  # Ensure unique IDs for each tab

        # Advanced Constructor
        if input_file in ['datacc2.xlsx', 'datavv2.xlsx']:
            vidPrefix = 'vv2'
            if input_file == 'datacc2.xlsx':
                vidPrefix = 'cc2'
                
            first_row = df.iloc[0]
            vlink = str(first_row.get("vlink", ""))
            all_data.append(
                "{\n"
                '  "type": "icn",\n'
                '  "role": "button",\n'
                '  "alt": "' + first_row.get("alt", "").replace('"', '\\"') + '",\n'
                '  "id": "icn' + sheet_suffix + '",\n'
                '  "content": "' + first_row.get("content", "").replace('"', '\\"') + '",\n'
                '  "left": "' + first_row.get("left", "").replace('"', '\\"') + '",\n'
                '  "top": "' + first_row.get("top", "").replace('"', '\\"') + '",\n'
                '  "vlink": "' + vlink + '",\n'
                '  "action": "openGroupGhost",\n'
                '  "target": "grp' + sheet_suffix + '"\n'
                '}'
            )

            group = "{\n" + \
                '  "type": "grp",\n' + \
                '  "id": "grp' + sheet_suffix + '",\n' + \
                '  "style": "grpAdv",\n' + \
                '  "left": "4em",\n' + \
                '  "top": "2em",\n' + \
                '  "width": "56em",\n' + \
                '  "height": "25em",\n' + \
                '  "visible": "false",\n' + \
                '  "children": [\n'

            children_data = []
            wordCount = 0
            for index, row in df.iterrows():
                if index == 0:
                    continue  # Skip the first row
                if row['type'] == 'vid':
                    child_str = '    {\n'
                    child_str += '      "type":"grp",'
                    child_str += '      "id": "videoGrp01",'
                    child_str += '      "style": "sdwVideo",'
                    child_str += '      "content": "",'
                    child_str += '      "track": "",'
                    child_str += '      "top": "1em",'
                    child_str += '      "left": "1em",'
                    child_str += '      "width": "20em",'
                    child_str += '      "height": "23em"'
                    child_str += '    }'
                    children_data.append(child_str)
                else:
                    child_str = '    {\n'
                    keys = list(row.keys())
                    valid_keys = [key for key in keys if pd.notna(row[key]) and row[key] != ""]
                    specialChar = False
                    for i, key in enumerate(valid_keys):
                        value = row[key]
                        
                        if key == 'style' and value == 'vector':
                            specialChar = True
                            # Add handling for vector styles
                            child_str += (
                                '      "type": "img",\n'
                                '      "style": "specialChar",\n'
                                '      "left": "88em",\n'
                                '      "top": "60em",\n'
                                '      "height": "9em",\n'
                                '      "width": "auto",\n'
                                '      "content": "' + row.get("content", "").replace('"', '\\"') + '"\n'
                            )
                        else:
                            if (specialChar==False):
                                child_str += '      "' + key + '": "' + str(value).replace('"', '\\"') + '"'

                            if i < len(valid_keys) - 1:
                                child_str += ',\n'
                            elif 'style' in row and row['style'] == 'words':
                                child_str += ',\n'
                            else:
                                child_str += '\n'
                    
                    if 'style' in row and row['style'] == 'words':
                        wordCount += 1
                        child_str += (
                            '      "children": [\n'
                            '        {\n'
                            '          "type": "icn",\n'
                            '          "role": "image",\n'
                            '          "content": "' + row.get("icn", "").replace('"', '\\"') + '",\n'
                            '          "left": "-4em",\n'
                            '          "top": "-1em",\n'
                            '          "height": "8em",\n'
                            '          "width": "8em"\n'
                            '        },\n'
                            '        {\n'
                            '          "type": "btn",\n'
                            '          "id": "",\n'
                            '          "style": "wAudPlay",\n'
                            '          "role": "button",\n'
                            '          "action": "playAudio",\n'
                            '          "target": "sample.wav",\n'
                            '          "right": "-4em",\n'
                            '          "top": "-1em",\n'
                            '          "height": "8em",\n'
                            '          "width": "8em"\n'
                            '        },\n'
                            '        {\n'
                            '          "type": "aud",\n'
                            '          "id": "",\n'
                            '          "style": "wAudPlayer",\n'
                            '          "role": "media",\n'
                            '          "action": "",\n'
                            '          "content": "'+row.get("content", "")+'",\n'
                            '          "right": "0em",\n'
                            '          "top": "0em",\n'
                            '          "height": "2em",\n'
                            '          "width": "8em"\n'
                            '        },\n'
                            '        {\n'
                            '          "type": "txt",\n'
                            '          "style": "vocab",\n'
                            '          "content": "' + row.get("text", "").replace('"', '\\"') + '",\n'
                            '          "left": "-.0em",\n'
                            '          "top": ".25em"\n'
                            '        }\n'
                            '      ]\n'
                        )

                    child_str += '    }'
                    children_data.append(child_str)
            
            group += ',\n'.join(children_data) + '\n'
            group += '  ]\n'
            group += '}\n'

            all_data.append(group)

        # Basic Constructor
        elif input_file in ['datacc.xlsx', 'datavv.xlsx']:
            # Handle the basic construction logic here
            row1 = df.iloc[0]
            row2 = df.iloc[1]    
            row3 = df.iloc[2]

            all_data.append('{\n'
                '  "type": "icn",\n'
                '  "role": "image",\n'
                '  "alt": "' + row1.get("alt", "").replace('"', '\\"') + '",\n'
                '  "id": "' + row1.get("alt", "").replace('"', '\\"') + '",\n'
                '  "content": "' + row1.get("content", "").replace('"', '\\"') + '",\n'
                '  "left": "' + row1.get("left", "").replace('"', '\\"') + '",\n'
                '  "top": "' + row1.get("top", "").replace('"', '\\"') + '",\n'
                '  "action": "toggleMulti",\n'
                '  "target": "grp' + sheet_suffix + '",\n'
                '  "children": [\n'
                '                   {'
                '                       "type": "btn",'
                '                       "id": "' + str(row2.get('id', 'default_id')) + '",'
                '                       "alt": "' + str(row2.get('alt', 'default_id')) + '",'
                '                       "name": "' + str(row2.get('alt', 'default_id')) + '",'
                '                       "style": "hotspot",'
                '                       "content": "",'
                '                       "left": "0",'
                '                       "top": "0em",'
                '                       "height": "9.5em",'
                '                       "width": "9.5em"'
                '                   },'
                '                   {'
                '                        "type": "grp",'
                '                        "id": "' + str(row3.get('id', 'default_id')) + '",'
                '                        "content": "' + str(row3.get('content', 'default_content')) + '",'
                '                        "alt": "' + str(row3.get('alt', 'default_id')) + '",'
                '                        "left": ".4em",'
                '                        "top": "11em",'
                '                        "width": "9em",'
                '                        "height": "12em",'
                '                        "visible": "false",'
                '                        "children": ['
                '                            {'
                '                                "type": "txt",'
                '                                "content": "' + str(row3.get('text', 'default_text')) + '",'
                '                                "left": "-.15em",'
                '                                "top": "-.15em"'
                '                            },'
                '                            {'
                '                                "type": "img",'
                '                                "style": "photo",'
                '                                "alt": "' + str(row3.get('alt', 'default_alt')) + '",'
                '                                "content": "' + str(row3.get('content', 'default_content')) + '",'
                '                                "left": ".5em",'
                '                                "top": "3.5em",'
                '                                "width": "7.5em",'
                '                                "height": "auto"'
                '                            }'
                '                        ]\n'
                '                    }\n'
                '                 ]\n'
                
                '}'
            )

    # Write to JSON file with utf-8 encoding
    with open(output_file, 'w', encoding='utf-8') as json_file:
        json_file.write(',\n'.join(all_data))

# Specify the input and output file paths for each Excel file
file_mappings = {
    "datavv.xlsx": "output1.json",
    "datavv2.xlsx": "output2.json",
    "datacc.xlsx": "output3.json",
    "datacc2.xlsx": "output4.json"
}

# Process each file
for input_file, output_file in file_mappings.items():
    excel_to_json_string(input_file, output_file)


Now we will insert the json snippets into the framework file.

In [2]:
def replace_placeholders(shell_file, output_files, output_file):
    # Open the shell file with the appropriate encoding
    with open(shell_file, 'r', encoding='utf-8') as file:
        shell_data = file.read()

    # Read the contents of the JSON snippets as text
    with open(output_files[0], 'r', encoding='utf-8') as file1, \
         open(output_files[1], 'r', encoding='utf-8') as file2, \
         open(output_files[2], 'r', encoding='utf-8') as file3, \
         open(output_files[3], 'r', encoding='utf-8') as file4:
        
        output1 = file1.read()
        output2 = file2.read()
        output3 = file3.read()
        output4 = file4.read()
    
    # Replace placeholders with the actual content from the JSON files
    shell_data = shell_data.replace("/*ADD output1.json here*/", output1)
    shell_data = shell_data.replace("/*ADD output2.json here*/", output2)
    shell_data = shell_data.replace("/*ADD output3.json here*/", output3)
    shell_data = shell_data.replace("/*ADD output4.json here*/", output4)
    
    # Write the final result to the output file with the correct encoding
    with open(output_file, 'w', encoding='utf-8') as file:
        file.write(shell_data)

# File paths
shell_file = '../data/soundwallShell.json'
output_files = ['output1.json', 'output2.json', 'output3.json', 'output4.json']
output_file = '../data/soundwall.json'

# Run the function
replace_placeholders(shell_file, output_files, output_file)
